In [ ]:
# 018 = no trigger markers found (0 files)
# 019 = too few trigger markers found (603) (3 files: 3, 300, 300)
# 020 = too few trigger markers found (300) (1 file)
# 035 = some issue with instrument marker file found (according to Oskari)
# 039 = too few trigger markers found (659) (1 file)
# 042 = too few trigger markers found (1119) (1 file)
# 044_rep = too few trigger markers found (959) (1 file)
# 045 = too few trigger markers found(282) (1 file)
# 045_rep = = too few trigger markers found (864) (1 file)
# 046 = too few trigger markers found (647) (1 file)
# 048 = too few trigger markers found(1076) (1 file)
# 049 = too few trigger markers found (1128) (1 file)
# 050 = too few trigger markers found (1114) (1 file)
# 123 = warning


# X = NORMAL, PERPENDICULAR TO COIL, INTO THE BRAIN
# Y = DIRECTION, HANDLE POSITION (REFERRED TO AS ORI, ORIENTATION, IN THESE SCRIPTS)
# Z = ORIENTATION RIGHTSIDE (NOT USED IN THESE SCRIPTS)
# P = POSITION, LOCATION OF THE COIL

# Subjects with unvalid NaN trials = 018 (100%), 019 (51.8%), 020 (78.5%), 039 (44%), 042 (6.9%), 044_rep (29%), 045 (74.6%), 045_rep (27.7%), 046 (46.5%), 048 (11.9%), 049 (5.8%), 050 (9.5%)
# Subjects with excessive NaNs, > 50% = 018, 019, 020, 045

In [ ]:
import scipy
import mne
import os
import shutil
import numpy as np
import mne
import scipy
import sys
from datetime import datetime

In [ ]:
def read_coil_control(subject, coil_control_path, distance_thresh, angle_distance_thresh):
      
    # Load trigger marker file
    coil_control = scipy.io.loadmat(coil_control_path, simplify_cells=True)['trigger_markers_remaining']['trigger_markers']
    n_trials = len(coil_control) # after preprocessing
     
    # Find valid (non NaN) trigger markers
    trigger_idx_notNaN = np.array([i for i in range(len(coil_control)) if str(coil_control[i]['stimulus_id']) != 'nan']) #some stimuli are not detected
    trigger_idx_NaN = np.array([i for i in range(len(coil_control)) if str(coil_control[i]['stimulus_id']) == 'nan']) 
    
    # Load instrument marker (target) file 
    if 'REFTEP_1' in coil_control_path:
        target_name = "target_marker"
    elif 'REFTEP_0' in coil_control_path:
        target_name = "instrument_marker"
    target = scipy.io.loadmat(coil_control_path,simplify_cells=True)['trigger_markers_remaining'][target_name]
          
    if any(sub in coil_control_path for sub in ['REFTEP_020']): #here the coil control was constantly off, maybe bad instrument marker,...so using median target
        target_pos = np.median([np.array(coil_control[i]['coil_pos']) for i in trigger_idx_notNaN],axis=0)
        target_normal = np.median([np.array(coil_control[i]['coil_normal']) for i in trigger_idx_notNaN],axis=0)
        target_ori = np.median([np.array(coil_control[i]['coil_dir']) for i in trigger_idx_notNaN],axis=0)
        legit_target_info = False
    else:
        target_pos = target['coil_pos']
        target_normal = target['coil_normal']
        target_ori = target['coil_dir'] # coil direction, y, coil handle
        legit_target_info = True
    print(target_pos, target_normal, target_ori)
    
    # Get coil position, normal and orientation (dir) for all trigger markers, with dummy values (target values) for NaN values
    # NaN trials get dummy deviations = 0
    coil_pos_all = [np.array(coil_control[i]['coil_pos']) if i in trigger_idx_notNaN else np.array(target_pos) for i in range(len(coil_control))]
    coil_normal_all = [np.array(coil_control[i]['coil_normal']) if i in trigger_idx_notNaN else np.array(target_normal) for i in range(len(coil_control))]
    coil_ori_all = [np.array(coil_control[i]['coil_dir']) if i in trigger_idx_notNaN else np.array(target_ori) for i in range(len(coil_control))]
  
    # Calculate Euclidean distances between coil position of all trigger markers to the coil position of the target marker
    coil_deviations_pos = np.linalg.norm(coil_pos_all - np.array(target_pos), axis=1) 
    trigger_idx_thresholded_pos = list(np.where(coil_deviations_pos <= distance_thresh)[0])
    
    # Calculate angle differences between coil normal of all trigger markers to the coil normal of the target marker
    coil_deviations_normal = get_angle_differences(coil_normal_all, target_normal)
    trigger_idx_thresholded_normal = list(np.where(coil_deviations_normal <= angle_distance_thresh)[0])

    # Calculate angle differences between coil orientation (direction, y) of all trigger markers to the coil orientation (direction, y) of the target marker
    coil_deviations_ori = get_angle_differences(coil_ori_all, target_ori)
    trigger_idx_thresholded_ori = list(np.where(coil_deviations_ori <= angle_distance_thresh)[0])
    
    # Get trials within coil position and orientation thresholds from target
    trigger_idx_thresholded_coil_only = np.intersect1d(np.intersect1d(trigger_idx_thresholded_ori, trigger_idx_thresholded_normal),trigger_idx_thresholded_pos) # includes NaN trials with dummy deviations = 0
    trigger_idx_thresholded = np.intersect1d(trigger_idx_thresholded_coil_only, trigger_idx_notNaN).astype(int) # removes original NaN trials
        
    # Get final coil deviations
    diff_pos = np.array([val if idx in trigger_idx_thresholded else np.nan for idx, val in enumerate(coil_deviations_pos)])
    diff_normal = np.array([val if idx in trigger_idx_thresholded else np.nan for idx, val in enumerate(coil_deviations_normal)])
    diff_ori = np.array([val if idx in trigger_idx_thresholded else np.nan for idx, val in enumerate(coil_deviations_ori)])

    # Sanity checks
    if min(diff_pos[trigger_idx_thresholded]) > distance_thresh or min(diff_normal[trigger_idx_thresholded]) > angle_distance_thresh or min(diff_ori[trigger_idx_thresholded]) > angle_distance_thresh:
        print("WARNING: some trials still pass thresholds")       
    if n_trials != len(diff_pos) != len(diff_normal) != len(diff_ori):
        print(f"WARNING: mismatch between total number of trials and diff outputs")
    ids = [np.array(coil_control[i]['stimulus_id']) for i in range(n_trials) if i in trigger_idx_notNaN]
    if len(ids) != len(np.unique(ids)):
        print("WARNING: mismatch NaN values")
    if subject[4] in ['1']: # all positions are recorded at Aalto so if there is a NaN then the position was not available in neuronavigation
        if len(diff_pos) != len(trigger_idx_notNaN):
            print(f"{len(diff_pos)-len(trigger_idx_notNaN)} nan stimuli found for {subject} that is not from Tuebingen, these are rejected")
            
    # Summary
    n_total = n_trials
    n_valid = len(trigger_idx_notNaN)
    n_kept = len(trigger_idx_thresholded)    
    n_rejected_total = n_trials - n_kept
    n_rejected_NaN = len(trigger_idx_NaN)
    n_rejected_deviations = n_valid - n_kept 
    n_rejected_pos = len(np.setdiff1d(trigger_idx_notNaN, trigger_idx_thresholded_pos))
    n_rejected_normal = len(np.setdiff1d(trigger_idx_notNaN, trigger_idx_thresholded_normal))
    n_rejected_ori = len(np.setdiff1d(trigger_idx_notNaN, trigger_idx_thresholded_ori))
    n_rejected_all3 = len(np.intersect1d(
        np.intersect1d(np.setdiff1d(trigger_idx_notNaN, trigger_idx_thresholded_pos), np.setdiff1d(trigger_idx_notNaN, trigger_idx_thresholded_normal)),
        np.setdiff1d(trigger_idx_notNaN, trigger_idx_thresholded_ori)
    ))
    mean_diff_pos = np.mean(coil_deviations_pos[trigger_idx_thresholded])
    mean_diff_normal = np.mean(coil_deviations_normal[trigger_idx_thresholded])
    mean_diff_ori = np.mean(coil_deviations_ori[trigger_idx_thresholded])
       
    print(f"[{subject}] Coil-deviation QC summary:")
    print(f"  - Total trials: {n_total}")
    print(f"  - Used legit instrument marker (target): {legit_target_info}")
    print(f"  - Valid trigger markers: {n_valid} ({(100*n_valid/n_total):.1f}%)")
    print(f"  - Total rejected (combined NaN and coil deviations) of all input trials: {n_rejected_total} ({(100*n_rejected_total/n_total):.1f}%)")
    print(f"  - Rejected due to NaN neuronavigation: {n_rejected_NaN} ({(100*n_rejected_NaN/n_total):.1f}%)")
    print(f"  - Rejected due to coil deviations: {n_rejected_deviations} "
        f"({(100*n_rejected_deviations/n_valid):.1f}% of valid)")
    print(f"    • Rejected due to Position threshold: {n_rejected_pos} "
        f"({(100*n_rejected_pos/n_valid):.1f}%)")
    print(f"    • Rejected due to Normal angle threshold: {n_rejected_normal} "
        f"({(100*n_rejected_normal/n_valid):.1f}%)")
    print(f"    • Rejected due to Direction angle threshold: {n_rejected_ori} "
        f"({(100*n_rejected_ori/n_valid):.1f}%)")
    print(f"    • Rejected by all three criteria: {n_rejected_all3} "
        f"({(100*n_rejected_all3/n_valid):.1f}%)")
    print(f"  - Trials kept for analysis: {n_kept} "
        f"({(100*n_kept/n_total):.1f}% of total, {(100*n_kept/n_valid):.1f}% of valid)")
    print(f"  - Mean Position deviation (kept): {mean_diff_pos:.1f} mm")
    print(f"  - Mean Normal deviation (kept): {mean_diff_normal:.1f}°")
    print(f"  - Mean Direction deviation (kept): {mean_diff_ori:.1f}°")

    # Save all coil control data as dictionary
    coil_drift_data = {
        "subject": subject,
        "n_trials_after_preprocessing": n_trials,
        "legit_target_info": legit_target_info,

        # Trial indices; PYTHON INDICES START AT 0, USE +1 FOR IN MATLAB
        "indices": {
            "kept": trigger_idx_thresholded,
            "kept_coil_only": trigger_idx_thresholded_coil_only,
            "kept_pos": trigger_idx_thresholded_pos,
            "kept_normal": trigger_idx_thresholded_normal,
            "kept_ori": trigger_idx_thresholded_ori,
            "valid": trigger_idx_notNaN,
            "nan": trigger_idx_NaN,
        },

        # Deviations (all trials, NaN where rejected)
        "deviations": {
            "pos": diff_pos,
            "normal": diff_normal,
            "ori": diff_ori,
        },

        # Raw deviations (before masking); trials with NaN marker info have deviations = 0
        "raw_deviations": {
            "pos": coil_deviations_pos,
            "normal": coil_deviations_normal,
            "ori": coil_deviations_ori,
        },
    }    
            
    return coil_drift_data #return the coil control parameters

def get_angle_differences(directions, target_direction):
    #get the angle differences of each stimulation relative to the target
    angle_list = []
    for compare in directions:
        dotprod = np.dot(compare,target_direction)
        magnitude_ori = np.linalg.norm(compare)
        magnitude_target_ori = np.linalg.norm(target_direction)
        angle = np.degrees(np.arccos(np.clip(dotprod/(magnitude_ori*magnitude_target_ori),-1.0,1.0)))
        angle_list.append(angle)
    return np.array(angle_list)

In [ ]:
# Start diary log file
log_file = f""
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()
sys.stdout = Tee(sys.stdout, open(log_file, 'a'))

# Specify output base path
DATAIN_BASE = ""
DATAOUT_BASE = ""

# Parameter settings
distance_thresh = 5 #np.inf
angle_distance_thresh = 10 #np.inf

# --- Accumulators ---
n_total_trials = []
n_valid_trials = []
n_kept_trials = []
pct_nan = []
pct_dev_rej = []
mean_pos = []
mean_normal = []
mean_ori = []
no_legit_coil_poses = 0

# Loop over subjects to calculate coil drifts
for subject in sorted(os.listdir(DATAIN_BASE)):
    if "REFTEP_" in subject: #if there are other files then do not look at them.
        print(subject)        
        if subject == "REFTEP_018" or subject == 'REFTEP_112' or subject == 'REFTEP_119':
            print("Skipping", subject, '\n')
            continue         
        
        coil_control_path = os.path.join(DATAIN_BASE, subject, 'stimulation_times', f'{subject}_stimulations_final.mat')
        print(coil_control_path)
        coil_drift_data = read_coil_control(subject, coil_control_path, distance_thresh, angle_distance_thresh)      
        scipy.io.savemat(os.path.join(DATAOUT_BASE, subject, 'stimulation_times', f'{subject}_coil_drift_data.mat'), coil_drift_data)     
            
        # Pull fields from dict
        n_trials = int(coil_drift_data["n_trials_after_preprocessing"])
        legit_target_info = coil_drift_data["legit_target_info"]
        idx_valid = np.array(coil_drift_data["indices"]["valid"]).astype(int)
        idx_nan = np.array(coil_drift_data["indices"]["nan"]).astype(int)
        idx_kept = np.array(coil_drift_data["indices"]["kept"]).astype(int)
        diff_pos = np.array(coil_drift_data["deviations"]["pos"], dtype=float)
        diff_normal = np.array(coil_drift_data["deviations"]["normal"], dtype=float)
        diff_ori = np.array(coil_drift_data["deviations"]["ori"], dtype=float)

        n_total = n_trials
        n_valid = len(idx_valid)
        n_nan = len(idx_nan)
        n_kept = len(idx_kept)
        n_dev_rej = n_valid - n_kept  # rejected by deviations among valid only

        # Append for group-level stats
        n_total_trials.append(n_total)
        n_valid_trials.append(n_valid)
        n_kept_trials.append(n_kept)

        pct_nan.append(0.0 if n_total == 0 else 100.0 * n_nan / n_total)
        pct_dev_rej.append(0.0 if n_valid == 0 else 100.0 * n_dev_rej / n_valid)

        # Per-subject targeting accuracies (means over KEPT trials)
        mean_pos.append(np.nanmean(diff_pos))
        mean_normal.append(np.nanmean(diff_normal))
        mean_ori.append(np.nanmean(diff_ori))

        if not legit_target_info:
            no_legit_coil_poses += 1
        print("\n")

# OVERALL AVERAGES ACROSS ALL SUBJECTS
print("=" * 50)
print("OVERALL STATISTICS ACROSS ALL SUBJECTS")
print("=" * 50)
print(f"Subjects processed: {len(n_total_trials)}")
print(f"No legit coil poses: {no_legit_coil_poses}")
print(f"Rejected due to NaN neuronavigation (of total trials): {np.mean(pct_nan):.1f} ± {np.std(pct_nan):.1f}%")
print(f"Rejected due to coil deviations (of valid trials): {np.mean(pct_dev_rej):.1f} ± {np.std(pct_dev_rej):.1f}%")
print("\nTargeting accuracies (mean over kept trials, per subject):")
print(f"Coil Position: {np.mean(mean_pos):.1f} ± {np.std(mean_pos):.1f} mm (median: {np.median(mean_pos):.1f})")
print(f"Coil Normal: {np.mean(mean_normal):.1f} ± {np.std(mean_normal):.1f}° (median: {np.median(mean_normal):.1f})")
print(f"Coil Direction: {np.mean(mean_ori):.1f} ± {np.std(mean_ori):.1f}° (median: {np.median(mean_ori):.1f})")
